In [ ]:
import streamlit as st
import pandas as pd
from datetime import datetime

st.title("Piyasa İlanları ve Filtreleme Paneli")
st.caption("Aktif veritabanındaki gayrimenkul ilanlarını kriterlerinize göre sorgulayabilirsiniz.")

if 'df' not in st.session_state:
    st.warning("Lütfen önce ana sayfayı (app.py) çalıştırınız.")
    st.stop()

df = st.session_state['df']

def mesafe_etiket(m):
    if m <= 800: return f"~{m} m (Çok Yakın)"
    if m <= 1500: return f"~{m} m (Yakın)"
    if m <= 3000: return f"~{m} m (Orta Mesafede)"
    return f"~{m} m (Uzak)"

def has_ozellik(lst, aranan):
    return any(aranan.lower() in o.lower() for o in lst)

kpi1, kpi2, kpi3, kpi4 = st.columns(4)
kpi1.metric("Toplam Kayıtlı İlan", f"{len(df):,} Adet")
kpi2.metric("Ortalama İlan Bedeli", f"{df['Fiyat'].mean():,.0f} TL")
kpi3.metric("Ortalama Birim Fiyat (m²)", f"{df['fiyat_m2'].mean():,.0f} TL")
kpi4.metric("Ortalama Konut Alanı", f"{df['m² (Brüt)'].mean():.0f} m²")

st.markdown("<br>", unsafe_allow_html=True)

with st.expander("Filtreleri Göster / Gizle", expanded=False):
    if st.button("Filtre Seçimlerini Sıfırla"):
        st.session_state["f_ilce"] = "Tüm İlçeler"
        st.session_state["f_mahalle"] = "Tüm Mahalleler"
        st.session_state["f_oda"] = "Tümü"
        st.session_state["f_m2"] = (int(df['m² (Brüt)'].min()), int(df['m² (Brüt)'].max()))
        st.session_state["f_butce"] = min(40_000_000, int(df['Fiyat'].max()))
        st.session_state["f_yas"] = (0, 30)
        st.session_state["f_ulasim"] = "Tümü"
        st.session_state["f_market"] = "Tümü"
        st.session_state["f_hastane"] = "Tümü"
        st.session_state["c_deprem"] = False
        st.session_state["c_kentsel"] = False
        st.session_state["c_site"] = False
        st.session_state["c_havuz"] = False
        st.session_state["c_asansor"] = False
        st.session_state["c_otopark"] = False
        st.session_state["c_ebeveyn"] = False
        st.session_state["c_dubleks"] = False
        st.session_state["c_bahce"] = False
        st.session_state["c_sifir"] = False
        st.rerun()

    f_col1, f_col2, f_col3 = st.columns(3)
    with f_col1:
        st.markdown("##### Lokasyon Parametreleri")
        st.selectbox("İl", ["İstanbul"], key="f_il_sabit")
        
        ilce_list = ["Tüm İlçeler"] + sorted(df['İlçe'].unique().tolist())
        selected_ilce = st.selectbox("İlçe", ilce_list, key="f_ilce")
        
        if selected_ilce == "Tüm İlçeler":
            mahalle_list = ["Tüm Mahalleler"] + sorted(df['Mahalle'].unique().tolist())
        else:
            mahalle_options = sorted(df[df['İlçe'] == selected_ilce]['Mahalle'].unique().tolist())
            mahalle_list = ["Tüm Mahalleler"] + mahalle_options if len(mahalle_options) > 0 else ["Kayıtlı Mahalle Yok"]

        selected_mahalle = st.selectbox("Mahalle", mahalle_list, key="f_mahalle")

    with f_col2:
        st.markdown("##### Yapı ve Yaş Özellikleri")
        selected_oda = st.selectbox("Oda Yapısı", ["Tümü"] + sorted(df['Oda Düzeni'].unique().tolist()), key="f_oda")
        min_m2, max_m2 = int(df['m² (Brüt)'].min()), int(df['m² (Brüt)'].max())
        m2_range = st.slider("Brüt Alan (m²)", min_m2, max_m2, (40, 350), key="f_m2")
        max_price = int(df['Fiyat'].max())
        budget = st.number_input("Üst Bütçe Limiti (TL)", 500_000, max_price, min(40_000_000, max_price), 500_000, key="f_butce")
        bina_yas_range = st.slider("Bina Yaşı Aralığı", 0, 50, (0, 30), key="f_yas")

    with f_col3:
        st.markdown("##### Konum ve Yakınlık")
        f_ulasim = st.selectbox("Toplu Ulaşım Mesafesi", ["Tümü", "1.5 km'den yakın", "3 km'den yakın"], key="f_ulasim")
        f_market = st.selectbox("Ticari Alan / AVM", ["Tümü", "1.5 km'den yakın"], key="f_market")
        f_hastane = st.selectbox("Sağlık Kuruluşu", ["Tümü", "2 km'den yakın"], key="f_hastane")

    st.markdown("##### Deprem Güvenliği ve Yapı Nitelikleri")
    d1, d2 = st.columns(2)
    with d1:
        f_deprem = st.checkbox("Deprem Yönetmeliğine Uygun / Radye Temel", key="c_deprem")
    with d2:
        f_kentsel = st.checkbox("Kentsel Dönüşüm / Güçlendirilmiş / Hasarsız", key="c_kentsel")

    st.markdown("##### Bina ve Konut Donatıları")
    o1, o2, o3, o4 = st.columns(4)
    with o1:
        f_site = st.checkbox("Site Yerleşimi", key="c_site")
        f_havuz = st.checkbox("Yüzme Havuzu", key="c_havuz")
    with o2:
        f_asansor = st.checkbox("Asansörlü", key="c_asansor")
        f_otopark = st.checkbox("Otopark / Kapalı Otopark", key="c_otopark")
    with o3:
        f_ebeveyn = st.checkbox("Ebeveyn Banyosu", key="c_ebeveyn")
        f_dubleks = st.checkbox("Dubleks Yapı", key="c_dubleks")
    with o4:
        f_bahce = st.checkbox("Bahçe Katı", key="c_bahce")
        f_sifir = st.checkbox("Sıfır / Yeni Bina", key="c_sifir")

selected_ilce = st.session_state.get("f_ilce", "Tüm İlçeler")
selected_mahalle = st.session_state.get("f_mahalle", "Tüm Mahalleler")
selected_oda = st.session_state.get("f_oda", "Tümü")
m2_range = st.session_state.get("f_m2", (int(df['m² (Brüt)'].min()), int(df['m² (Brüt)'].max())))
budget = st.session_state.get("f_butce", min(40_000_000, int(df['Fiyat'].max())))
bina_yas_range = st.session_state.get("f_yas", (0, 30))
f_ulasim = st.session_state.get("f_ulasim", "Tümü")
f_market = st.session_state.get("f_market", "Tümü")
f_hastane = st.session_state.get("f_hastane", "Tümü")
f_deprem = st.session_state.get("c_deprem", False)
f_kentsel = st.session_state.get("c_kentsel", False)
f_site = st.session_state.get("c_site", False)
f_havuz = st.session_state.get("c_havuz", False)
f_asansor = st.session_state.get("c_asansor", False)
f_otopark = st.session_state.get("c_otopark", False)
f_ebeveyn = st.session_state.get("c_ebeveyn", False)
f_dubleks = st.session_state.get("c_dubleks", False)
f_bahce = st.session_state.get("c_bahce", False)
f_sifir = st.session_state.get("c_sifir", False)

filtered = df.copy()
if selected_ilce != "Tüm İlçeler":
    filtered = filtered[filtered['İlçe'] == selected_ilce]
if selected_mahalle != "Tüm Mahalleler":
    filtered = filtered[filtered['Mahalle'] == selected_mahalle]
if selected_oda != "Tümü":
    filtered = filtered[filtered['Oda Düzeni'] == selected_oda]

filtered = filtered[
    (filtered['m² (Brüt)'] >= m2_range[0]) & 
    (filtered['m² (Brüt)'] <= m2_range[1]) & 
    (filtered['Fiyat'] <= budget) &
    (filtered['bina_yasi'] >= bina_yas_range[0]) &
    (filtered['bina_yasi'] <= bina_yas_range[1])
]

if f_ulasim == "1.5 km'den yakın": filtered = filtered[filtered['ulasim_mesafe_m'] <= 1500]
elif f_ulasim == "3 km'den yakın": filtered = filtered[filtered['ulasim_mesafe_m'] <= 3000]
if f_market == "1.5 km'den yakın": filtered = filtered[filtered['market_mesafe_m'] <= 1500]
if f_hastane == "2 km'den yakın": filtered = filtered[filtered['hastane_mesafe_m'] <= 2000]

if f_deprem: filtered = filtered[filtered['Ozellikler'].apply(lambda x: has_ozellik(x, "Deprem Yönetmeliğine Uygun"))]
if f_kentsel: filtered = filtered[filtered['Ozellikler'].apply(lambda x: has_ozellik(x, "Kentsel Dönüşüm / Güçlendirilmiş"))]
if f_site: filtered = filtered[filtered['Ozellikler'].apply(lambda x: has_ozellik(x, "Site içinde"))]
if f_havuz: filtered = filtered[filtered['Ozellikler'].apply(lambda x: has_ozellik(x, "Havuz"))]
if f_asansor: filtered = filtered[filtered['Ozellikler'].apply(lambda x: has_ozellik(x, "Asansör"))]
if f_otopark: filtered = filtered[filtered['Ozellikler'].apply(lambda x: has_ozellik(x, "Otopark"))]
if f_ebeveyn: filtered = filtered[filtered['Ozellikler'].apply(lambda x: has_ozellik(x, "Ebeveyn Banyolu"))]
if f_dubleks: filtered = filtered[filtered['Ozellikler'].apply(lambda x: has_ozellik(x, "Dubleks"))]
if f_bahce: filtered = filtered[filtered['Ozellikler'].apply(lambda x: has_ozellik(x, "Bahçeli"))]
if f_sifir: filtered = filtered[filtered['Ozellikler'].apply(lambda x: has_ozellik(x, "Sıfır"))]

arama = st.text_input("Metin Arama (İlan Başlığı / İlçe / Mahalle)", placeholder="Örn: 3+1, Kadıköy, radye temel, 5 yıllık...", key="arama_input")
if arama:
    metin = arama.lower().strip()
    filtered = filtered[
        filtered['İlan Başlığı'].str.lower().str.contains(metin, na=False) |
        filtered['İlçe'].str.lower().str.contains(metin, na=False) |
        filtered['Mahalle'].str.lower().str.contains(metin, na=False)
    ]

st.markdown(f"### Sorgulama Sonuçları ({len(filtered)} İlan Listelendi)")
if len(filtered) == 0:
    st.warning("Belirtilen ilçe veya arama kriterlerine uygun kayıt bulunamadı.")
else:
    show_df = filtered.head(40).copy()
    show_df['m² Birim Fiyatı'] = show_df['fiyat_m2'].apply(lambda x: f"{x:,.0f} TL")
    show_cols = ['İlan Başlığı', 'İlçe', 'Mahalle', 'Oda Düzeni', 'bina_yasi', 'm² (Brüt)', 'm² Birim Fiyatı', 'Fiyat']
    
    event = st.dataframe(
        show_df[show_cols].style.format({'Fiyat': '{:,.0f} TL', 'm² (Brüt)': '{:.0f}'}),
        column_config={
            "bina_yasi": st.column_config.NumberColumn("Bina Yaşı", format="%d Yıl")
        },
        use_container_width=True, height=300,
        on_select="rerun", selection_mode="single-row", key="df_selection"
    )

    export_df = filtered.copy()
    export_df['m² Birim Fiyatı'] = export_df['fiyat_m2'].apply(lambda x: f"{x:,.0f} TL")
    export_cols = ['İlan Başlığı', 'İl', 'İlçe', 'Mahalle', 'Oda Düzeni', 'bina_yasi', 'm² (Brüt)', 'm² Birim Fiyatı', 'Fiyat']
    valid_export_cols = [c for c in export_cols if c in export_df.columns]
    
    csv_data = export_df[valid_export_cols].to_csv(index=False, encoding='utf-8-sig')
    st.download_button("Listelenen İlanları CSV Olarak İndir", data=csv_data, file_name=f"Gayrimenkul_Ilan_Listesi_{datetime.now().strftime('%Y%m%d')}.csv", mime="text/csv")

    selected_rows = event.get("selection", {}).get("rows", [])
    st.divider()
    
    row = show_df.iloc[selected_rows[0]] if selected_rows else show_df.iloc[0]

    st.markdown(f"### Seçili Kayıt Detayı: {row['İlan Başlığı']}")
    st.caption(f"Lokasyon: **İstanbul / {row['İlçe']} / {row['Mahalle']}** | Tip: **{row['Oda Düzeni']}** | Bina Yaşı: **{row['bina_yasi']} Yıl** | Alan: **{row['m² (Brüt)']} m²** | Satış Bedeli: **{row['Fiyat']:,.0f} TL**")
    
    c1, c2, c3 = st.columns(3)
    c1.metric("Toplu Ulaşım Noktası", mesafe_etiket(row['ulasim_mesafe_m']))
    c2.metric("Ticari Merkez / AVM", mesafe_etiket(row['market_mesafe_m']))
    c3.metric("Sağlık Kuruluşu", mesafe_etiket(row['hastane_mesafe_m']))

    ozellikler = row['Ozellikler']
    if ozellikler:
        st.markdown("**Tespit Edilen Yapı ve Güvenlik Nitelikleri:**")
        etiket_html = " ".join([f'<span class="ozellik-etiket">{o}</span>' for o in ozellikler])
        st.markdown(etiket_html, unsafe_allow_html=True)